In [ ]:
"""
GANs are models that generate new, realistic data by learning from existing data. Introduced by Ian Goodfellow in 2014,
they enable machines to create content like images, videos and music.

Architecture of GAN
GAN consists of two neural networks the generator and the discriminator trained adversarially, where the generator tries to fool the discriminator and
the discriminator tries to distinguish real from fake data.
"""

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
#  Defining Image Transformations

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
# Loading the CIFAR-10 Dataset

# if steel have not downloaded dataset change download flag to True
train_dataset = datasets.CIFAR10(root='./data',\
              train=True, download=False, transform=transform)
dataloader = torch.utils.data.DataLoader(train_dataset, \
                                batch_size=32, shuffle=True)

In [9]:
# Step Defining GAN Hyperparameters

latent_dim = 100
lr = 0.0002
beta1 = 0.5
beta2 = 0.999
# if have gpu change num_epochs to 10
num_epochs = 1

In [10]:
# Building the Generator

class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (128, 8, 8)),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128, momentum=0.78),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64, momentum=0.78),
            nn.ReLU(),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        return img

In [11]:
#  Building the Discriminator

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.25),
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
        nn.ZeroPad2d((0, 1, 0, 1)),
        nn.BatchNorm2d(64, momentum=0.82),
        nn.LeakyReLU(0.25),
        nn.Dropout(0.25),
        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
        nn.BatchNorm2d(128, momentum=0.82),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.25),
        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(256, momentum=0.8),
        nn.LeakyReLU(0.25),
        nn.Dropout(0.25),
        nn.Flatten(),
        nn.Linear(256 * 5 * 5, 1),
        nn.Sigmoid()
    )

    def forward(self, img):
        validity = self.model(img)
        return validity

In [12]:
# Initializing GAN Components
generator = Generator(latent_dim).to(device)
discriminator = Discriminator().to(device)

adversarial_loss = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters()\
                         , lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters()\
                         , lr=lr, betas=(beta1, beta2))

In [13]:
# Training the GAN

for epoch in range(num_epochs):
    for i, batch in enumerate(dataloader):
       
        real_images = batch[0].to(device) 
       
        valid = torch.ones(real_images.size(0), 1, device=device)
        fake = torch.zeros(real_images.size(0), 1, device=device)
       
        real_images = real_images.to(device)

        optimizer_D.zero_grad()
       
        z = torch.randn(real_images.size(0), latent_dim, device=device)
      
        fake_images = generator(z)

        real_loss = adversarial_loss(discriminator\
                                     (real_images), valid)
        fake_loss = adversarial_loss(discriminator\
                                     (fake_images.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
    
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
      
        gen_images = generator(z)
        
        g_loss = adversarial_loss(discriminator(gen_images), valid)
        g_loss.backward()
        optimizer_G.step()
       
        if (i + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}]\
                        Batch {i+1}/{len(dataloader)} "
                f"Discriminator Loss: {d_loss.item():.4f} "
                f"Generator Loss: {g_loss.item():.4f}"
            )
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            z = torch.randn(16, latent_dim, device=device)
            generated = generator(z).detach().cpu()
            grid = torchvision.utils.make_grid(generated,\
                                        nrow=4, normalize=True)
            plt.imshow(np.transpose(grid, (1, 2, 0)))
            plt.axis("off")
            plt.show()

Epoch [1/1]                        Batch 100/1563 Discriminator Loss: 0.5967 Generator Loss: 1.3104
Epoch [1/1]                        Batch 200/1563 Discriminator Loss: 0.8786 Generator Loss: 0.7728
Epoch [1/1]                        Batch 300/1563 Discriminator Loss: 0.4498 Generator Loss: 1.2975
Epoch [1/1]                        Batch 400/1563 Discriminator Loss: 0.5155 Generator Loss: 1.4342
Epoch [1/1]                        Batch 500/1563 Discriminator Loss: 0.7310 Generator Loss: 1.0354
Epoch [1/1]                        Batch 600/1563 Discriminator Loss: 0.5827 Generator Loss: 1.1410
Epoch [1/1]                        Batch 700/1563 Discriminator Loss: 0.6400 Generator Loss: 0.9287
Epoch [1/1]                        Batch 800/1563 Discriminator Loss: 0.5793 Generator Loss: 1.2861
Epoch [1/1]                        Batch 900/1563 Discriminator Loss: 0.7190 Generator Loss: 0.7843
Epoch [1/1]                        Batch 1000/1563 Discriminator Loss: 0.6949 Generator Loss: 0.7813